# RAG Foundations — Interactive Walkthrough

> **Part 1 of 6 of the RAG Full Stack Series**  
> Article: [Why Your LLM Needs a Library: RAG Fundamentals for the Pragmatic Engineer](https://linkedin.com/in/rezkyauliapratama)

This notebook walks through every component of a **Retrieval-Augmented Generation (RAG)** pipeline step-by-step — from raw markdown documents to a grounded LLM answer.

---

## What We're Building

```
┌─────────────────────────────────────────────────────────────────┐
│  INDEX (run once)                                               │
│                                                                 │
│  data/*.md  ──▶  Chunker  ──▶  Embedder  ──▶  Vector Store     │
└─────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────┐
│  QUERY (per user question)                                      │
│                                                                 │
│  Question  ──▶  Embedder  ──▶  Retriever  ──▶  Generator  ──▶  Answer
└─────────────────────────────────────────────────────────────────┘
```

**5 components, 2 phases.** Each cell below corresponds to one component.

---

## Prerequisites

Run this notebook from the **repo root** using UV:

```bash
uv sync --extra dev
cp .env.example .env   # fill in LLM_PROVIDER and API key
uv run jupyter lab
```

Then open: `rag-series/01-rag-foundations/notebooks/01_rag_foundations.ipynb`

---
## 0. Setup — Environment & Path

In [ ]:
import sys
import os
from pathlib import Path

# Resolve repo root (two levels up from this notebook) and module root
NOTEBOOK_DIR = Path().resolve()
MODULE_ROOT  = NOTEBOOK_DIR.parent          # 01-rag-foundations/
REPO_ROOT    = MODULE_ROOT.parent.parent    # genai-in-practice/
DATA_DIR     = MODULE_ROOT / "data"

# Add both roots to sys.path so both `src.*` and `_shared.*` imports resolve
for p in [str(MODULE_ROOT), str(REPO_ROOT)]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"Module root : {MODULE_ROOT}")
print(f"Repo root   : {REPO_ROOT}")
print(f"Data dir    : {DATA_DIR}")
print(f"Data files  : {[f.name for f in sorted(DATA_DIR.glob('*.md'))]}")

In [ ]:
# Load .env from repo root
from dotenv import load_dotenv
load_dotenv(REPO_ROOT / ".env")

from _shared.config import settings
print(f"LLM provider       : {settings.llm_provider}")
print(f"Embedding provider : {settings.embedding_provider}")
print(f"Log level          : {settings.log_level}")

---
## 1. Chunker — Load & Split the Knowledge Base

**What it does:** Reads every `.md` file in `data/` and splits the text into overlapping chunks that fit within an embedding model's context window.

**Key design decisions:**
- `chunk_size=512` chars ≈ 1 FAQ entry — small enough to be specific, large enough to carry context
- `chunk_overlap=50` prevents a sentence straddling a boundary from being silently dropped
- Separator priority: `\n## ` → `\n\n` → `\n` → `' '` — always prefer to split at markdown section headings first

In [ ]:
from src.chunker import load_documents, chunk_documents

# Step 1a — Load raw markdown files
docs = load_documents(str(DATA_DIR))

print(f"Loaded {len(docs)} documents:")
for doc in docs:
    word_count = len(doc['text'].split())
    print(f"  • {doc['source']}  ({word_count} words)")

In [ ]:
# Step 1b — Split documents into chunks
chunks = chunk_documents(docs, chunk_size=512, chunk_overlap=50)

print(f"Total chunks: {len(chunks)}")
print()

# Distribution across source documents
from collections import Counter
dist = Counter(c['metadata']['source'] for c in chunks)
for source, count in sorted(dist.items()):
    print(f"  {source:20s}  {count} chunks")

In [ ]:
# Inspect a sample chunk to understand the schema
sample = chunks[3]
print("Chunk schema:")
print(f"  doc_id   : {sample['doc_id']}")
print(f"  metadata : {sample['metadata']}")
print(f"  len(content): {len(sample['content'])} chars")
print()
print("Content:")
print("-" * 60)
print(sample['content'])

---
## 2. Embedder — Text → Vectors

**What it does:** Converts text into dense numerical vectors where **distance ≈ semantic similarity**. Two sentences with the same meaning land close together in vector space even if they share no words.

**Provider options** (set via `EMBEDDING_PROVIDER` in `.env`):

| Provider | Model | Cost | Dimension |
|---|---|---|---|
| `huggingface` | `all-MiniLM-L6-v2` | Free, runs on CPU | 384 |
| `openai` | `text-embedding-3-small` | ~$0.00002/1K tokens | 1536 |

> ⚠️ **Critical rule:** documents and queries **must use the same model**. Mixing models produces garbage scores.

In [ ]:
from src.embedder import Embedder

embedder = Embedder()
print(f"Embedding provider: {embedder.provider}")

# Quick smoke-test: embed two sentences
test_texts = [
    "how do I block my card?",
    "the capital of France is Paris",
]
test_vecs = embedder.embed(test_texts)
print(f"Vector dimension : {len(test_vecs[0])}")
print(f"First 5 values   : {[round(v, 4) for v in test_vecs[0][:5]]}")

In [ ]:
# Embed ALL chunks — this is the indexing step
import time

texts = [c['content'] for c in chunks]

t0 = time.perf_counter()
embeddings = embedder.embed(texts)
elapsed = time.perf_counter() - t0

print(f"Embedded {len(embeddings)} chunks in {elapsed:.2f}s")
print(f"Embedding matrix shape: ({len(embeddings)}, {len(embeddings[0])})")

In [ ]:
# Visualise semantic similarity between two sentences
import numpy as np

def cosine_sim(a, b):
    return float(np.dot(a, b))  # vectors are already L2-normalised

pairs = [
    ("how do I block my card?",    "how to freeze my debit card"),    # same intent
    ("how do I block my card?",    "BI-FAST transfer fee"),            # different topic
    ("how do I block my card?",    "what is the capital of France?"),  # unrelated
]

print("Cosine similarity between sentence pairs:\n")
for s1, s2 in pairs:
    v1, v2 = embedder.embed([s1, s2])
    sim = cosine_sim(v1, v2)
    bar = "█" * int(sim * 40)
    print(f"  {sim:.3f}  {bar}")
    print(f"         A: \"{s1}\"")
    print(f"         B: \"{s2}\"")
    print()

---
## 3. Vector Store — Store & Search

**What it does:** Stores all chunk embeddings and answers `similarity_search(query_vector)` by computing dot products across the entire matrix.

**The "magic" of semantic search in one line:**
```python
scores = embedding_matrix @ query_vector
```
Because vectors are L2-normalised, dot product == cosine similarity. Highest score = most similar meaning.

**Two implementations — same interface:**

| Store | Module | Use case |
|---|---|---|
| `InMemoryVectorStore` | `vector_store.py` | Learning, prototyping, < 10K chunks |
| `QdrantVectorStore` | `qdrant_store.py` | Production, persistence, millions of chunks |

In [ ]:
from src.vector_store import InMemoryVectorStore

store = InMemoryVectorStore()
store.upsert_documents(chunks, embeddings)

print(f"Stored {len(chunks)} chunks in InMemoryVectorStore")
print(f"Embedding matrix shape: {store._matrix.shape}")

In [ ]:
# Raw similarity search (no threshold yet)
query = "how do I block my card?"
query_vec = embedder.embed([query])[0]

raw_results = store.similarity_search(query_vec, top_k=5)

print(f"Top-5 results for: \"{query}\"\n")
for i, r in enumerate(raw_results, 1):
    print(f"[{i}] score={r['score']:.4f}  source={r['metadata']['source']}")
    print(f"    {r['content'][:120].strip()}...")
    print()

In [ ]:
# Show what happens with an out-of-scope query — scores drop but results still return!
# This is WHY we need a threshold in the Retriever (next section).
oos_query = "what is the capital of France?"
oos_vec = embedder.embed([oos_query])[0]
oos_results = store.similarity_search(oos_vec, top_k=3)

print(f"Top-3 results for out-of-scope query: \"{oos_query}\"\n")
for i, r in enumerate(oos_results, 1):
    print(f"[{i}] score={r['score']:.4f}  source={r['metadata']['source']}")
    print(f"    {r['content'][:120].strip()}...")
    print()

print("⚠️  Without a threshold, the store always returns results — even for irrelevant queries.")

---
## 3b. Level Up — Qdrant Vector Store

The `QdrantVectorStore` is a **drop-in replacement** for `InMemoryVectorStore` — same `upsert_documents()` and `similarity_search()` interface.

**Extra capabilities over in-memory:**
- **Persistence:** index survives process restarts
- **Scale:** HNSW ANN indexes — millisecond search at millions of vectors
- **Filtering:** metadata filter enforced at DB level (e.g. `source="compliance.md"`)
- **Idempotent ingestion:** `uuid5` deterministic IDs — re-indexing updates, never duplicates

**To run this cell, start Qdrant first:**
```bash
docker run -p 6333:6333 -v $(pwd)/qdrant_storage:/qdrant/storage qdrant/qdrant
```

In [ ]:
# ── OPTIONAL: swap to Qdrant ──────────────────────────────────────────────────
# Requires Qdrant running on localhost:6333 (see docker command above)

USE_QDRANT = False  # flip to True after starting Docker

if USE_QDRANT:
    from src.qdrant_store import QdrantVectorStore
    qdrant_store = QdrantVectorStore(collection="rag_foundations_nb", dim=len(embeddings[0]))
    qdrant_store.upsert_documents(chunks, embeddings)
    print("Qdrant store ready. Testing similarity search...")

    results = qdrant_store.similarity_search(query_vec, top_k=3)
    for r in results:
        print(f"  score={r['score']:.4f}  {r['content'][:80].strip()}...")

    # Metadata filter: restrict search to compliance.md only
    print("\nFiltered search (compliance.md only):")
    filtered = qdrant_store.similarity_search(query_vec, top_k=3, source="compliance.md")
    for r in filtered:
        print(f"  score={r['score']:.4f}  src={r['metadata']['source']}  {r['content'][:80].strip()}...")
else:
    print("Skipping Qdrant cell. Set USE_QDRANT = True and start Docker to run.")

---
## 4. Retriever — Top-K + Relevance Gate

**What it does:** Wraps the vector store with a `score_threshold` filter. If no chunk clears the bar, an **empty list** is returned — and the generator will refuse to answer without ever calling the LLM.

**Why a separate module from the store?**  
Raw similarity search *always* returns `top_k` results, even when none are relevant. The threshold is the architectural defence against out-of-scope queries.

**Threshold calibration guide:**
- MiniLM scores for relevant matches: typically `0.30 – 0.70`
- OpenAI `text-embedding-3-small` scores: typically `0.60 – 0.90`
- Calibrate by scoring 10 "should match" + 10 "should not match" queries and placing the threshold in the gap

In [ ]:
from src.retriever import Retriever

retriever = Retriever(
    embedder=embedder,
    store=store,
    top_k=5,
    score_threshold=0.3,
)
print(f"Retriever ready  top_k={retriever.top_k}  threshold={retriever.score_threshold}")

In [ ]:
# Test the 5 suggested queries from the README
test_queries = [
    ("how do I block my card?",              "Direct match"),
    ("how to freeze my debit card",           "Semantic match — different words, same intent"),
    ("BI-FAST limit",                         "Partial phrase match"),
    ("what documents for new account",        "Multi-concept query"),
    ("what is the capital of France?",        "Out-of-scope — should return 0 chunks"),
]

for query, label in test_queries:
    results = retriever.retrieve(query)
    print(f"[{label}]")
    print(f"  Query   : \"{query}\"")
    print(f"  Chunks  : {len(results)} passed threshold")
    if results:
        top = results[0]
        print(f"  Top hit : score={top['score']:.4f}  source={top['metadata']['source']}")
        print(f"            {top['content'][:100].strip()}...")
    print()

In [ ]:
# Score distribution across threshold values — helps calibrate your threshold
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6]
in_scope  = "how do I block my card?"
out_scope = "what is the capital of France?"

in_vec  = embedder.embed([in_scope])[0]
out_vec = embedder.embed([out_scope])[0]

in_scores  = sorted([r['score'] for r in store.similarity_search(in_vec,  top_k=3)], reverse=True)
out_scores = sorted([r['score'] for r in store.similarity_search(out_vec, top_k=3)], reverse=True)

print("Score distributions (top-3):")
print(f"  In-scope  '{in_scope[:40]}': {[round(s,3) for s in in_scores]}")
print(f"  Out-scope '{out_scope[:40]}': {[round(s,3) for s in out_scores]}")
print()
print("Threshold sensitivity (in-scope | out-scope chunks passing):")
for t in thresholds:
    r_in  = Retriever(embedder, store, top_k=5, score_threshold=t)
    r_out = Retriever(embedder, store, top_k=5, score_threshold=t)
    n_in  = len(r_in.retrieve(in_scope))
    n_out = len(r_out.retrieve(out_scope))
    flag  = "  ✅" if n_in > 0 and n_out == 0 else ("  ⚠️" if n_in == 0 else "")
    print(f"  threshold={t:.1f}  in-scope={n_in} chunks  out-scope={n_out} chunks{flag}")

---
## 5. Generator — Augment Prompt & Call LLM

**What it does:** Builds an augmented prompt from retrieved chunks and calls the LLM. If no chunks were retrieved, it returns a fallback message **without touching the LLM** — zero tokens spent.

**Two-layer hallucination defence:**

| Layer | Where | Mechanism |
|---|---|---|
| Architectural | `retriever.py` | `score_threshold` → empty list → no LLM call |
| Behavioural | `generator.py` | System prompt: "Answer ONLY from context" |

Either layer alone fails sometimes. Together they fail rarely.

In [ ]:
from src import generator

# Inspect the prompt template and system instructions
print("SYSTEM INSTRUCTIONS:")
print("-" * 60)
print(generator.SYSTEM_INSTRUCTIONS)
print()
print("PROMPT TEMPLATE:")
print("-" * 60)
print(generator.PROMPT_TEMPLATE)

In [ ]:
# Show the fully-built prompt for an in-scope query
question = "how do I block my card?"
retrieved_chunks = retriever.retrieve(question)
context_block = generator.build_context(retrieved_chunks)

augmented_prompt = generator.PROMPT_TEMPLATE.format(
    context=context_block,
    question=question,
)

from _shared.utils import estimate_tokens
print(f"Retrieved chunks  : {len(retrieved_chunks)}")
print(f"Prompt length     : {len(augmented_prompt)} chars  (~{estimate_tokens(augmented_prompt)} tokens)")
print()
print("AUGMENTED PROMPT (first 800 chars):")
print("-" * 60)
print(augmented_prompt[:800])
print("...")

In [ ]:
# Generate the answer
answer = generator.generate(question, retrieved_chunks)
print(f"Q: {question}")
print()
print(f"A: {answer}")

In [ ]:
# Test the fallback path — out-of-scope query, no LLM call
oos_question = "what is the capital of France?"
oos_chunks = retriever.retrieve(oos_question)

print(f"Chunks retrieved for out-of-scope query: {len(oos_chunks)}")
fallback = generator.generate(oos_question, oos_chunks)
print(f"\nQ: {oos_question}")
print(f"A: {fallback}")
print()
print("✅ No LLM call was made — generator short-circuited at the empty-chunks check.")

---
## 6. RAGPipeline — Full End-to-End

The `RAGPipeline` class wires all 5 components into two clean methods:
- `pipeline.index(data_dir)` → load → chunk → embed → store
- `pipeline.query(question)` → embed → retrieve → augment → generate

In [ ]:
from src.pipeline import RAGPipeline

pipeline = RAGPipeline(top_k=5, score_threshold=0.3)

t0 = time.perf_counter()
n_chunks = pipeline.index(str(DATA_DIR))
elapsed = time.perf_counter() - t0

print(f"Indexed {n_chunks} chunks in {elapsed:.2f}s — pipeline ready.")

In [ ]:
# Run all 5 test queries end-to-end
test_queries_e2e = [
    "how do I block my card?",
    "how to freeze my debit card",
    "BI-FAST limit",
    "what documents for new account",
    "what is the capital of France?",
]

for q in test_queries_e2e:
    print(f"{'─'*70}")
    print(f"Q: {q}")
    answer = pipeline.query(q)
    print(f"A: {answer}")
    print()

In [ ]:
# Interactive mode — type your own questions
# Run this cell, type a question, press Enter. Type 'exit' to stop.

print("Interactive RAG — type a question (or 'exit' to stop):\n")
while True:
    question = input("Your question: ").strip()
    if question.lower() in ("exit", "quit", ""):
        print("Exiting.")
        break
    print(f"\n{pipeline.query(question)}\n")

---
## 7. Shared Utilities — Config, LLM Client, Helpers

In [ ]:
# _shared/config.py — type-safe settings via Pydantic
from _shared.config import settings

print("Settings (from .env):")
print(f"  llm_provider       = {settings.llm_provider}")
print(f"  embedding_provider = {settings.embedding_provider}")
print(f"  log_level          = {settings.log_level}")
print(f"  openai_api_key     = {'SET' if settings.openai_api_key else 'NOT SET'}")
print(f"  gemini_api_key     = {'SET' if settings.gemini_api_key else 'NOT SET'}")
print(f"  groq_api_key       = {'SET' if settings.groq_api_key else 'NOT SET'}")
print(f"  deepseek_api_key   = {'SET' if settings.deepseek_api_key else 'NOT SET'}")

In [ ]:
# _shared/llm_client.py — provider-agnostic chat()
# OpenAI, Gemini, and Groq all expose OpenAI-compatible endpoints.
# Switching provider = one .env change, zero code changes.
from _shared.llm_client import chat

response = chat(
    system="You are a helpful assistant. Keep answers under 2 sentences.",
    user="What is RAG in one sentence?",
)
print(f"LLM ({settings.llm_provider}): {response}")

In [ ]:
# _shared/utils.py — logger, timer decorator, token estimator
from _shared.utils import get_logger, timer, estimate_tokens

# Logger
logger = get_logger("notebook")
logger.info("Logger is working")

# Timer decorator
@timer
def slow_function(n: int) -> list:
    return list(range(n))

slow_function(1_000_000)

# Token estimator
sample_prompt = "Tell me about the BI-FAST transfer limits for mobile banking."
est = estimate_tokens(sample_prompt)
print(f"\nEstimated tokens for sample prompt: ~{est}")

---
## 8. Knowledge Base — Data Inspection

In [ ]:
# Print the full contents of all 4 knowledge base files
for path in sorted(DATA_DIR.glob("*.md")):
    print(f"{'═'*70}")
    print(f"  {path.name}")
    print(f"{'═'*70}")
    print(path.read_text(encoding="utf-8"))
    print()

---
## 9. Summary — What We Built

| Component | Module | Key Insight |
|---|---|---|
| **Chunker** | `src/chunker.py` | Markdown-aware splitting; overlap prevents boundary loss |
| **Embedder** | `src/embedder.py` | Same model for docs AND queries — never mix |
| **Vector Store** | `src/vector_store.py` | `matrix @ query_vec` == cosine similarity |
| **Retriever** | `src/retriever.py` | `score_threshold` is the gate against hallucination |
| **Generator** | `src/generator.py` | Empty-chunks check first — no tokens wasted on refusals |
| **Pipeline** | `src/pipeline.py` | `index()` + `query()` — swap store in one line |

**Two-layer hallucination defence:**
1. **Architectural** — `Retriever.score_threshold` → empty list → generator returns fallback with zero LLM calls
2. **Behavioural** — system prompt: *"Answer ONLY from the provided context"*

---

## Next Steps

- **Swap to Qdrant** for persistence and production-grade search (see Section 3b)
- **Tune `score_threshold`** using the calibration cell in Section 4
- **Extend the knowledge base** — add more `.md` files to `data/` and re-run `pipeline.index()`
- **Watch for Part 2** of the RAG Full Stack Series